# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Samad-17/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #2 — The Content Performance Curve (health score peaks at 61-90 days, decays by 271-365 days, "recovers" at 365+)

Methodology question: The 365+ "recovery" bucket is explicitly flagged by the paper itself as needing a narrower reading, and I want to push on why. The health score used here is a composite built partly from impressions and position — both of which are also used elsewhere in the paper as separate outcome variables. Where does the age-bucket sample for 365+ come from — is it a survivor-biased subset (only pages that are still active and tracked at 365+ days, meaning already-failed old pages have dropped out of the sample entirely)? The paper acknowledges this survivor-bias risk directly for the related 365+ × 361+ cell in Finding #8, but doesn't explicitly quantify it here for the main curve. My question: does the validation design confirm the 365+ uptick reflects refreshed pages recovering, or could it partly reflect that only the strongest old pages survive long enough to still be in the "active content" sample at all (impressions_90d > 0)?

Finding: ML Appendix — Feature Importance for Health Score (Random Forest)

Methodology question: The paper is admirably upfront that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal" — avg_position (43%) and impressions (32%) dominate the importance ranking, and both are literally components of the health score formula (impressions worth 30 pts, position worth 30 pts). My question: given this, does an 80/20 holdout split (as described in the methodology section) meaningfully validate anything here beyond "the model can reconstruct a formula from its own known ingredients"? A holdout split protects against overfitting to noise, but it doesn't protect against a feature literally being a direct component of the label — that's a structural, not a sample-size, issue. Would the feature importance ranking look meaningfully different (and more informative) if health score's own direct components were excluded from the model's inputs, leaving only genuinely independent signals like word_count, content_age_days, or days_since_update?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used a client-holdout split, but let's show the "before/after" explicitly — comparing it against a naive random row split (which allows the same client's pages to appear in both train and test), to demonstrate why the grouped split matters.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ["HF_TOKEN"]}');""")
BASE = "hf://datasets/FlyRank/internship-warehouse"

raw = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position,
           ga4_sessions, ga4_engaged_sessions
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
raw["report_date"] = pd.to_datetime(raw["report_date"])

early = raw[raw["report_date"].dt.day <= 15]
late = raw[raw["report_date"].dt.day > 15]

early_agg = early.groupby(["content_hash_id","client_hash_id"]).agg(
    impressions=("gsc_impressions","sum"), clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean"), sessions=("ga4_sessions","mean"),
    engaged_sessions=("ga4_engaged_sessions","mean")
).reset_index()
late_agg = late.groupby(["content_hash_id","client_hash_id"]).agg(late_clicks=("gsc_clicks","sum")).reset_index()

data = early_agg.merge(late_agg, on=["content_hash_id","client_hash_id"], how="inner")
data = data[data["impressions"] >= 50].copy()
data["label"] = (data["late_clicks"] < data["clicks"]).astype(int)
data["ctr"] = (data["clicks"] / data["impressions"]).fillna(0)
data["avg_position"] = data["avg_position"].fillna(100)
data["sessions"] = data["sessions"].fillna(0)
data["engaged_sessions"] = data["engaged_sessions"].fillna(0)

features = ["impressions","clicks","avg_position","ctr","sessions","engaged_sessions"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
# BEFORE: naive random row split (no client grouping — leakier)
train_naive, test_naive = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label"])

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                   class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf_naive.fit(train_naive[features], train_naive["label"])
probs_naive = rf_naive.predict_proba(test_naive[features])[:,1]

naive_p20 = precision_at_k(probs_naive, test_naive["label"].values, 20)
naive_p50 = precision_at_k(probs_naive, test_naive["label"].values, 50)
print("BEFORE (naive random split) — Precision@20:", round(naive_p20,3), "Precision@50:", round(naive_p50,3))

BEFORE (naive random split) — Precision@20: 0.85 Precision@50: 0.86


In [3]:
# AFTER: honest client-holdout split (same as Week 5)
rng = np.random.default_rng(42)
clients = data["client_hash_id"].unique()
test_clients = set(rng.choice(clients, size=max(1, int(len(clients)*0.2)), replace=False))

train_honest = data[~data["client_hash_id"].isin(test_clients)]
test_honest = data[data["client_hash_id"].isin(test_clients)]

rf_honest = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                    class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf_honest.fit(train_honest[features], train_honest["label"])
probs_honest = rf_honest.predict_proba(test_honest[features])[:,1]

honest_p20 = precision_at_k(probs_honest, test_honest["label"].values, 20)
honest_p50 = precision_at_k(probs_honest, test_honest["label"].values, 50)
print("AFTER (client-holdout split) — Precision@20:", round(honest_p20,3), "Precision@50:", round(honest_p50,3))

AFTER (client-holdout split) — Precision@20: 0.9 Precision@50: 0.78


Before (naive random split): Precision@20 = 0.85, Precision@50 = 0.86.
After (client-holdout split): Precision@20 = 0.90, Precision@50 = 0.78.

The result here is mixed rather than a clean drop or clean match, which is itself an honest finding worth reporting rather than smoothing over. Precision@20 actually rose slightly under the honest split (0.85 → 0.90), while Precision@50 dropped meaningfully (0.86 → 0.78).

The Precision@50 drop is the more informative signal: it suggests that under the naive random split, the model was partly benefiting from seeing some of a client's pages during training and being evaluated on other pages from that same client in the test set — letting it pick up client-specific quirks (a particular client's typical CTR range, content style, etc.) rather than a signal that generalizes across clients. Once client-holdout removes that overlap, performance at the deeper cutoff (top 50) is more honestly represented and comes in lower.

The small rise at Precision@20 is likely noise given the smaller sample size at that cutoff (only 20 rows), rather than evidence the honest split is somehow "better" — with such a small K, a couple of individual rows swinging the wrong way can shift the number either direction. The Precision@50 comparison is the more reliable read here, and it points toward: some of my Week-5 model's reported strength was inflated by leakage-adjacent data structure (same-client rows in both train and test), not purely a generalizable pattern.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Repeating the Week-3 leakage hunt on my final feature set: impressions, clicks, avg_position, ctr, sessions, engaged_sessions — none of these reference the late-window label or any FlyRank product decision flag (not present in this release). The feature window (days 1-15) and label window (days 16-31) don't overlap, confirmed by the empty set intersection above. One caveat carried over from Week 5: because the label is late_clicks < early_clicks, pages with higher early clicks are somewhat more likely to see a relative drop by construction (regression toward the mean) — not classic feature leakage, but a labeling-design limitation worth flagging honestly.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no target-window or product-flag columns are in the feature set
print("Features used:", features)
print("Any label-derived terms present?", any("label" in f or "late" in f for f in features))

# Confirm feature window and label window don't overlap in time
print("Feature window: days 1-15")
print("Label window: days 16-31")
print("Overlap:", set(range(1,16)) & set(range(16,32)))

Features used: ['impressions', 'clicks', 'avg_position', 'ctr', 'sessions', 'engaged_sessions']
Any label-derived terms present? False
Feature window: days 1-15
Label window: days 16-31
Overlap: set()


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original bold claim (from Week 5 write-up): "Random forest reached 0.90 precision@20, proving it reliably predicts which pages will decline."

Rewritten, safe version: "Under client-holdout validation, the random forest's top 20 ranked pages were observed to align with the decline label in 90% of cases on this dataset slice — a directional signal supporting its use as a decision-support ranking, not a guarantee of future decline for any individual page, and not evidence of what causes decline."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.